# 05 — Evaluate Saved Model Outputs

This notebook evaluates model responses generated by:

- `01_closed_models_base.ipynb`
- `02_closed_models_variants.ipynb`
- `03_open_models_base.ipynb`
- `04_open_models_variants.ipynb`

No benchmark language model is queried here.

The evaluation focuses on three separate questions:

1. **Answer correctness** — Is the final answer correct?
2. **Context-grounded reasoning faithfulness** — Are the reasoning claims supported by the supplied context?
3. **Confidence calibration** — Does the model's stated confidence correspond to actual correctness?

Automatic evaluators are not assumed to be ground truth.
They are first checked against human-annotated Turkish examples.

The historical evaluation implementation is preserved separately and is not
silently reused in this corrected evaluation pipeline.

## Evaluation workflow

The notebook has four stages:

### `prepare_pilot`

Create a small blinded human-validation pilot.

Purpose:
- check whether the annotation instructions are clear,
- inspect reasoning segmentation,
- identify unexpected cases before the formal validation study.

### `prepare_validation`

After the pilot is reviewed, create separate:

- development set,
- held-out test set.

The development set is used to select automatic evaluators and the semantic
similarity threshold.

The held-out test set is used only to measure the performance of the frozen
selection.

### `validate_evaluators`

Compare candidate automatic evaluators against human judgments.

Candidate answer-similarity evaluators:

- Jina Embeddings v3
- multilingual MPNet

Candidate NLI evaluators:

- mDeBERTa XNLI
- XLM-R XNLI

Only one primary semantic evaluator and one primary NLI evaluator are selected
for the full benchmark.

### `full`

Apply the validated and frozen evaluation configuration to every saved model
output.

The resulting evaluated datasets are later analyzed in
`06_results_and_analysis.ipynb`.

## Setup

In [1]:
%pip install -q \
    pandas \
    numpy \
    scikit-learn \
    "transformers==4.56.2" \
    sentence-transformers \
    accelerate \
    stanza \
    huggingface_hub \
    einops

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 794.2/794.2 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 14.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have hugging

In [2]:
from pathlib import Path

import json
import sys

import pandas as pd

## Project location

The notebook can be used from either:

- a cloned local/GitHub repository,
- a Google Drive copy in Colab.

In [3]:
USE_GOOGLE_DRIVE = True

DRIVE_PROJECT_PATH = Path(
    "/content/drive/MyDrive/turkish-llm-reasoning"
)


if USE_GOOGLE_DRIVE:

    from google.colab import drive

    drive.mount("/content/drive")

    PROJECT_ROOT = DRIVE_PROJECT_PATH

else:

    current = Path.cwd().resolve()

    PROJECT_ROOT = None

    for candidate in [
        current,
        *current.parents,
    ]:

        if (
            (candidate / "notebooks").exists()
            and
            (candidate / "src").exists()
            and
            (candidate / "data").exists()
        ):

            PROJECT_ROOT = candidate
            break


    if PROJECT_ROOT is None:

        raise FileNotFoundError(
            "Project root could not be found. "
            "Run this notebook inside the repository "
            "or enable Google Drive mode."
        )


SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:

    sys.path.insert(
        0,
        str(SRC_DIR),
    )


print("Project root:")
print(PROJECT_ROOT)

Mounted at /content/drive
Project root:
/content/drive/MyDrive/turkish-llm-reasoning


In [4]:
from turkish_reasoning.evaluation import (
    atomic_save_csv,
    atomic_save_json,
    load_generation_results,
    prepare_evaluation_data,
    run_full_evaluation,
    save_full_evaluation,
    validate_nli_evaluators,
    validate_semantic_evaluators,
)

from turkish_reasoning.human_validation import (
    load_labeled_validation_set,
    prepare_pilot_sets,
    prepare_validation_sets,
)

## Configuration

Normal experiment use should require changes mainly in this section.

`SOURCE_RUN_LABEL` identifies the generation run produced by notebooks 01–04.

`EVALUATION_LABEL` identifies this evaluation version.

The formal human-validation sample sizes are deliberately left undefined until
the pilot has been reviewed.

In [5]:
# ============================================================
# MAIN CONFIGURATION
# ============================================================

SOURCE_RUN_LABEL = "smoke_v1"

EVALUATION_LABEL = "smoke_eval_v5"


# ------------------------------------------------------------
# Current evaluation stage
# ------------------------------------------------------------

STAGE = "preview_evaluators"

VALID_STAGES = {
    "prepare_pilot",
    "prepare_validation",
    "validate_evaluators",
    "full",
    "preview_evaluators",
}


# ------------------------------------------------------------
# Pilot
# ------------------------------------------------------------

ANSWER_PILOT_SIZE = 12

REASONING_PILOT_OUTPUTS = 8


# ------------------------------------------------------------
# Formal human validation
#
# These values will be decided AFTER reviewing the pilot.
# ------------------------------------------------------------

ANSWER_DEV_SIZE = None

ANSWER_TEST_SIZE = None

REASONING_DEV_OUTPUTS = None

REASONING_TEST_OUTPUTS = None


# ------------------------------------------------------------
# Human annotation
# ------------------------------------------------------------

MIN_ANNOTATORS = 2


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_SEED = 20260926


# ------------------------------------------------------------
# Final safety gate
#
# Keep False until evaluator validation has been reviewed.
# ------------------------------------------------------------

APPROVE_FULL_EVALUATION = False


if STAGE not in VALID_STAGES:

    raise ValueError(
        f"Unknown STAGE: {STAGE}"
    )


print("Stage:", STAGE)
print("Source run:", SOURCE_RUN_LABEL)
print("Evaluation:", EVALUATION_LABEL)

Stage: preview_evaluators
Source run: smoke_v1
Evaluation: smoke_eval_v5


In [6]:
EVALUATION_ROOT = (
    PROJECT_ROOT
    / "results"
    / "evaluated"
    / EVALUATION_LABEL
)

VALIDATION_DIR = (
    EVALUATION_ROOT
    / "evaluator_validation"
)

FINAL_DIR = (
    EVALUATION_ROOT
    / "final"
)


EVALUATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print("Evaluation directory:")
print(EVALUATION_ROOT)

Evaluation directory:
/content/drive/MyDrive/turkish-llm-reasoning/results/evaluated/smoke_eval_v5


## Load generated results

The saved outputs from notebooks 01–04 are loaded and matched back to the
canonical datasets.

The loader verifies the saved dataset identity before evaluation.

Generation errors remain explicitly marked and are not silently converted into
ordinary model answers.

In [7]:
records, source_manifest = (
    load_generation_results(
        project_root=PROJECT_ROOT,
        source_run_label=SOURCE_RUN_LABEL,
    )
)


print(
    "Total saved model outputs:",
    len(records),
)


display(
    records.groupby(
        [
            "model_access",
            "input_type",
            "model_label",
        ],
        dropna=False,
    )
    .size()
    .rename("records")
    .reset_index()
)

Total saved model outputs: 40


,model_access,input_type,model_label,records
0,closed,base,gpt-4o-mini,5
1,closed,variants,gpt-4o-mini,15
2,open,base,qwen2.5-7b-instruct,5
3,open,variants,qwen2.5-7b-instruct,15


## Prepare evaluation data

Before using any learned evaluator, conservative deterministic checks are
performed.

For example:

- `25` vs `25` → deterministic match
- `25.0` vs `25` → numeric match

Failure to match these rules does **not** mean that an answer is incorrect.

It means that semantic equivalence still needs to be evaluated.

In [8]:
prepared = prepare_evaluation_data(
    records
)


summary = pd.DataFrame(
    {
        "metric": [
            "all records",
            "successful generations",
            "deterministic matches",
            "need semantic evaluation",
        ],

        "count": [
            len(prepared),

            int(
                prepared[
                    "generation_success"
                ].sum()
            ),

            int(
                prepared[
                    "rule_based_positive"
                ].sum()
            ),

            int(
                prepared[
                    "needs_semantic_evaluation"
                ].sum()
            ),
        ],
    }
)


display(summary)

,metric,count
0,all records,40
1,successful generations,40
2,deterministic matches,15
3,need semantic evaluation,19


## Human validation

Automatic evaluators are validated against human judgments before they are
used on the full benchmark.

### Answer-equivalence task

Annotators see:

- context,
- question,
- gold answer,
- model answer.

They do **not** see the model identity or confidence.

Labels:

- `equivalent`
- `not_equivalent`
- `ambiguous`

### Reasoning-support task

Model reasoning is split into sentence-level claims.

Annotators see:

- context,
- question,
- one reasoning claim.

Labels:

- `supported`
- `contradicted`
- `nei`

`nei` means **not enough information**.

Pilot, development and held-out test sets are kept separate.

In [9]:
if STAGE == "prepare_pilot":

    pilot = prepare_pilot_sets(
        records_df=prepared,
        evaluation_root=EVALUATION_ROOT,
        answer_size=ANSWER_PILOT_SIZE,
        reasoning_output_size=(
            REASONING_PILOT_OUTPUTS
        ),
        random_seed=RANDOM_SEED,
        download_stanza_if_missing=True,
    )


    print("Pilot prepared.")

    print(
        "Answer items:",
        pilot["answer_item_count"],
    )

    print(
        "Reasoning outputs:",
        pilot["reasoning_output_count"],
    )

    print(
        "Reasoning claims:",
        pilot["reasoning_claim_count"],
    )

    print()
    print("Created files:")

    for name, path in pilot["files"].items():

        print(
            f"- {name}: "
            f"{path.relative_to(PROJECT_ROOT)}"
        )

In [10]:
import transformers
print(transformers.__version__)

4.56.2


### Formal development and held-out test sets

This stage is run only after the pilot has been reviewed.

Pilot outputs are excluded from the formal validation sets.

The development and test sets are mutually exclusive.

The exact sample sizes will be chosen after examining the pilot rather than
being guessed in advance.

In [11]:
if STAGE == "prepare_validation":

    required_sizes = {
        "ANSWER_DEV_SIZE":
            ANSWER_DEV_SIZE,

        "ANSWER_TEST_SIZE":
            ANSWER_TEST_SIZE,

        "REASONING_DEV_OUTPUTS":
            REASONING_DEV_OUTPUTS,

        "REASONING_TEST_OUTPUTS":
            REASONING_TEST_OUTPUTS,
    }


    missing = [
        name
        for name, value
        in required_sizes.items()
        if value is None
    ]


    if missing:

        raise ValueError(
            "Formal validation sizes have not "
            "been decided yet:\n"
            + "\n".join(missing)
        )


    validation_sets = (
        prepare_validation_sets(
            records_df=prepared,
            evaluation_root=EVALUATION_ROOT,

            answer_dev_size=(
                ANSWER_DEV_SIZE
            ),

            answer_test_size=(
                ANSWER_TEST_SIZE
            ),

            reasoning_dev_outputs=(
                REASONING_DEV_OUTPUTS
            ),

            reasoning_test_outputs=(
                REASONING_TEST_OUTPUTS
            ),

            random_seed=RANDOM_SEED,

            download_stanza_if_missing=True,
        )
    )


    print(
        "Formal human-validation "
        "sets prepared."
    )

    display(
        pd.DataFrame(
            validation_sets[
                "counts"
            ]
        ).T
    )

## Validate automatic evaluators

This stage requires completed human annotations for both:

- development data,
- held-out test data.

Evaluator selection is made **only on the development set**.

The held-out test set is not used to:

- choose an evaluator,
- choose a semantic threshold,
- tune the method.

### Answer correctness candidates

- Jina Embeddings v3
- multilingual MPNet

### Reasoning-support candidates

- mDeBERTa XNLI
- XLM-R XNLI

One primary semantic evaluator and one primary NLI evaluator are selected.

Exact Hugging Face model revisions are saved so that the evaluation can be
reproduced later.

In [12]:
if STAGE == "validate_evaluators":

    # ========================================================
    # LOAD HUMAN GOLD DATA
    # ========================================================

    answer_dev, answer_dev_agreement = (
        load_labeled_validation_set(
            records_df=prepared,
            evaluation_root=EVALUATION_ROOT,
            task="answer",
            split="dev",
            minimum_annotators=MIN_ANNOTATORS,
        )
    )


    answer_test, answer_test_agreement = (
        load_labeled_validation_set(
            records_df=prepared,
            evaluation_root=EVALUATION_ROOT,
            task="answer",
            split="test",
            minimum_annotators=MIN_ANNOTATORS,
        )
    )


    reasoning_dev, reasoning_dev_agreement = (
        load_labeled_validation_set(
            records_df=prepared,
            evaluation_root=EVALUATION_ROOT,
            task="reasoning",
            split="dev",
            minimum_annotators=MIN_ANNOTATORS,
        )
    )


    reasoning_test, reasoning_test_agreement = (
        load_labeled_validation_set(
            records_df=prepared,
            evaluation_root=EVALUATION_ROOT,
            task="reasoning",
            split="test",
            minimum_annotators=MIN_ANNOTATORS,
        )
    )


    # ========================================================
    # SEMANTIC EVALUATOR VALIDATION
    # ========================================================

    (
        semantic_selection,
        semantic_dev_report,
        semantic_test_report,
    ) = validate_semantic_evaluators(
        answer_dev,
        answer_test,
    )


    # ========================================================
    # NLI EVALUATOR VALIDATION
    # ========================================================

    (
        nli_selection,
        nli_dev_report,
        nli_test_report,
    ) = validate_nli_evaluators(
        reasoning_dev,
        reasoning_test,
    )


    # ========================================================
    # FREEZE THE DEVELOPMENT-SET SELECTION
    # ========================================================

    validated_selection = {
        "selection_basis":
            "human_annotated_development_set",

        "semantic":
            semantic_selection,

        "nli":
            nli_selection,

        "human_agreement": {
            "answer_dev":
                answer_dev_agreement,

            "answer_test":
                answer_test_agreement,

            "reasoning_dev":
                reasoning_dev_agreement,

            "reasoning_test":
                reasoning_test_agreement,
        },
    }


    # ========================================================
    # SAVE VALIDATION REPORTS
    # ========================================================

    atomic_save_csv(
        semantic_dev_report,
        VALIDATION_DIR
        / "semantic_dev_report.csv",
    )

    atomic_save_csv(
        semantic_test_report,
        VALIDATION_DIR
        / "semantic_test_report.csv",
    )

    atomic_save_csv(
        nli_dev_report,
        VALIDATION_DIR
        / "nli_dev_report.csv",
    )

    atomic_save_csv(
        nli_test_report,
        VALIDATION_DIR
        / "nli_test_report.csv",
    )

    atomic_save_json(
        validated_selection,
        VALIDATION_DIR
        / "validated_selection.json",
    )


    print(
        "Semantic evaluator — development set"
    )

    display(
        semantic_dev_report
    )


    print(
        "Semantic evaluator — held-out test"
    )

    display(
        semantic_test_report
    )


    print(
        "NLI evaluator — development set"
    )

    display(
        nli_dev_report
    )


    print(
        "NLI evaluator — held-out test"
    )

    display(
        nli_test_report
    )


    print()
    print("Selected semantic evaluator:")

    print(
        json.dumps(
            semantic_selection,
            ensure_ascii=False,
            indent=2,
        )
    )


    print()
    print("Selected NLI evaluator:")

    print(
        json.dumps(
            nli_selection,
            ensure_ascii=False,
            indent=2,
        )
    )

## Full evaluation

This stage is intentionally locked until the human-validation and automatic
evaluator-validation results have been reviewed.

The full evaluation uses the previously frozen evaluator configuration.

It produces:

### Answer correctness

- deterministic exact / numeric matches,
- validated semantic equivalence for unresolved successful answers.

### Context-grounded reasoning faithfulness

Each reasoning sentence is evaluated against the supplied context using the
validated NLI evaluator.

The main saved measurements include:

- context support rate,
- contradiction rate,
- neutral / not-enough-information rate,
- mean entailment probability.

No arbitrary weighted combination of semantic similarity and NLI is used.

### Confidence calibration

Per-answer Brier scores are prepared here.

Model-level calibration statistics and reliability plots are calculated later
in `06_results_and_analysis.ipynb`.

In [13]:
if STAGE == "full":

    if not APPROVE_FULL_EVALUATION:

        raise RuntimeError(
            "Full evaluation is locked.\n"
            "Review evaluator validation first, "
            "then set:\n\n"
            "APPROVE_FULL_EVALUATION = True"
        )


    selection_path = (
        VALIDATION_DIR
        / "validated_selection.json"
    )


    if not selection_path.exists():

        raise FileNotFoundError(
            "Validated evaluator configuration "
            "was not found:\n"
            f"{selection_path}"
        )


    with open(
        selection_path,
        "r",
        encoding="utf-8",
    ) as file:

        validated_selection = json.load(
            file
        )


    evaluated, claims, audit = (
        run_full_evaluation(
            prepared_records=prepared,
            selection=validated_selection,
            download_stanza_if_missing=True,
        )
    )


    audit["evaluation_label"] = (
        EVALUATION_LABEL
    )

    audit["source_run_label"] = (
        SOURCE_RUN_LABEL
    )


    saved_files = save_full_evaluation(
        evaluated_records=evaluated,
        claim_level=claims,
        manifest_df=source_manifest,
        audit=audit,
        output_dir=FINAL_DIR,
    )


    print(
        "Full evaluation completed."
    )

    print()
    print("Saved files:")

    for name, path in saved_files.items():

        print(
            f"- {name}: "
            f"{path.relative_to(PROJECT_ROOT)}"
        )

## Evaluation outputs

The final evaluated results are stored under:

`results/evaluated/<evaluation-label>/`

The directory separates three types of information:

### `human_validation/`

Human-annotation templates, private mappings, completed annotations and
adjudication records.

### `evaluator_validation/`

Development/test reports and the frozen evaluator configuration.

### `final/`

Full benchmark evaluation outputs:

- `base_evaluated.csv`
- `variants_evaluated.csv`
- `claim_level_evaluation.csv`
- `source_manifest.csv`
- `evaluation_metadata.json`

Statistical comparisons, base-versus-variant analyses, calibration curves and
publication figures are intentionally handled in
`06_results_and_analysis.ipynb`.

## Smoke evaluator preview

This stage applies all candidate automatic evaluators to the saved smoke
outputs so their behavior can be inspected before formal human validation.

It does not:

- select a winning evaluator,
- determine a semantic threshold,
- create publication results,
- replace formal human validation.

Outputs are stored under:

`results/evaluated/<evaluation-label>/evaluator_preview/`

In [14]:
# ============================================================
# SMOKE EVALUATOR PREVIEW
# ============================================================

if STAGE == "preview_evaluators":

    from turkish_reasoning.evaluation import (
        _semantic_scores,
        _nli_scores,
        _build_claims,
    )

    PREVIEW_DIR = EVALUATION_ROOT / "evaluator_preview"
    PREVIEW_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    # ========================================================
    # 1. FINAL ANSWER — SEMANTIC EVALUATORS
    # ========================================================

    semantic_preview = (
        prepared[
            prepared["needs_semantic_evaluation"]
        ]
        .copy()
        .reset_index(drop=True)
    )

    print(
        "Answers requiring semantic evaluation:",
        len(semantic_preview),
    )

    jina_scores, jina_revision = _semantic_scores(
        "jina_v3",
        semantic_preview["model_answer"].astype(str).tolist(),
        semantic_preview["gold_answer"].astype(str).tolist(),
    )

    semantic_preview["jina_similarity"] = jina_scores

    mpnet_scores, mpnet_revision = _semantic_scores(
        "multilingual_mpnet",
        semantic_preview["model_answer"].astype(str).tolist(),
        semantic_preview["gold_answer"].astype(str).tolist(),
    )

    semantic_preview["mpnet_similarity"] = mpnet_scores

    semantic_columns = [
        column
        for column in [
            "record_uid",
            "record_id",
            "base_question_id",
            "input_type",
            "variant_type",
            "model_access",
            "model_label",
            "model_id",
            "question",
            "gold_answer",
            "model_answer",
            "jina_similarity",
            "mpnet_similarity",
        ]
        if column in semantic_preview.columns
    ]

    semantic_preview = semantic_preview[
        semantic_columns
    ].copy()

    semantic_path = (
        PREVIEW_DIR
        / "semantic_preview.csv"
    )

    atomic_save_csv(
        semantic_preview,
        semantic_path,
    )

    print()
    print("Semantic preview:")
    display(semantic_preview)

    # ========================================================
    # 2. REASONING — NLI EVALUATORS
    # ========================================================

    reasoning_preview = _build_claims(
        prepared,
        download_stanza_if_missing=True,
    ).reset_index(drop=True)

    print()
    print(
        "Reasoning claims:",
        len(reasoning_preview),
    )

    mdeberta_scores, mdeberta_revision, mdeberta_audit = (
        _nli_scores(
            "mdeberta_xnli",
            reasoning_preview["context"].astype(str).tolist(),
            reasoning_preview["claim_text"].astype(str).tolist(),
        )
    )

    mdeberta_scores = mdeberta_scores.rename(
        columns={
            "nli_label":
                "mdeberta_label",
            "entailment_probability":
                "mdeberta_entailment_probability",
            "neutral_probability":
                "mdeberta_neutral_probability",
            "contradiction_probability":
                "mdeberta_contradiction_probability",
        }
    )

    xlmr_scores, xlmr_revision, xlmr_audit = (
        _nli_scores(
            "xlmr_xnli",
            reasoning_preview["context"].astype(str).tolist(),
            reasoning_preview["claim_text"].astype(str).tolist(),
        )
    )

    xlmr_scores = xlmr_scores.rename(
        columns={
            "nli_label":
                "xlmr_label",
            "entailment_probability":
                "xlmr_entailment_probability",
            "neutral_probability":
                "xlmr_neutral_probability",
            "contradiction_probability":
                "xlmr_contradiction_probability",
        }
    )

    reasoning_preview = pd.concat(
        [
            reasoning_preview.reset_index(drop=True),
            mdeberta_scores.reset_index(drop=True),
            xlmr_scores.reset_index(drop=True),
        ],
        axis=1,
    )

    reasoning_path = (
        PREVIEW_DIR
        / "reasoning_nli_preview.csv"
    )

    atomic_save_csv(
        reasoning_preview,
        reasoning_path,
    )

    print()
    print("Reasoning NLI preview:")
    display(reasoning_preview)

    # ========================================================
    # 3. PREVIEW METADATA
    # ========================================================

    preview_metadata = {
        "source_run_label": SOURCE_RUN_LABEL,
        "evaluation_label": EVALUATION_LABEL,

        "purpose": (
            "Smoke-only evaluator preview; "
            "no evaluator or threshold is selected."
        ),

        "semantic_answer_count":
            len(semantic_preview),

        "reasoning_claim_count":
            len(reasoning_preview),

        "semantic_evaluators": {
            "jina_v3": {
                "revision": jina_revision,
            },
            "multilingual_mpnet": {
                "revision": mpnet_revision,
            },
        },

        "nli_evaluators": {
            "mdeberta_xnli": {
                "revision": mdeberta_revision,
                "length_audit": mdeberta_audit,
            },
            "xlmr_xnli": {
                "revision": xlmr_revision,
                "length_audit": xlmr_audit,
            },
        },

        "human_validated": False,
        "semantic_threshold_selected": False,
        "primary_evaluator_selected": False,
    }

    metadata_path = (
        PREVIEW_DIR
        / "preview_metadata.json"
    )

    atomic_save_json(
        preview_metadata,
        metadata_path,
    )

    print()
    print("Saved evaluator preview files:")
    print("-", semantic_path.relative_to(PROJECT_ROOT))
    print("-", reasoning_path.relative_to(PROJECT_ROOT))
    print("-", metadata_path.relative_to(PROJECT_ROOT))

Answers requiring semantic evaluation: 19


modules.json:   0%|          | 0.00/378 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/464 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

custom_st.py: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of the model checkpoint at joeddav/xlm-roberta-large-xnli were not used when initializing XLMRobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).



Reasoning NLI preview:


,record_uid,record_id,input_type,model_access,model_label,model_id,category,difficulty,claim_index,claim_text,context,mdeberta_label,mdeberta_entailment_probability,mdeberta_neutral_probability,mdeberta_contradiction_probability,xlmr_label,xlmr_entailment_probability,xlmr_neutral_probability,xlmr_contradiction_probability
0,989752d2aea04acec4601ecc,TR-1-018,base,closed,gpt-4o-mini,gpt-4o-mini,temporal_reasoning,easy,1,"Öncelikle, Matematik dersi 08.30'da başlamış v...",Sabah saat 08.30'da ilk ders zili çalan 3-A sı...,entailment,0.998858,0.000909,0.000233,entailment,0.999519,0.000169,0.000312
1,989752d2aea04acec4601ecc,TR-1-018,base,closed,gpt-4o-mini,gpt-4o-mini,temporal_reasoning,easy,2,Bu durumda Matematik dersi 09.10'da biter.,Sabah saat 08.30'da ilk ders zili çalan 3-A sı...,contradiction,0.071816,0.012507,0.915677,neutral,0.005459,0.975097,0.019444
2,989752d2aea04acec4601ecc,TR-1-018,base,closed,gpt-4o-mini,gpt-4o-mini,temporal_reasoning,easy,3,"Ardından 10 dakika teneffüs yapılacak, bu da 0...",Sabah saat 08.30'da ilk ders zili çalan 3-A sı...,contradiction,0.006212,0.290670,0.703117,entailment,0.507618,0.284871,0.207511
3,989752d2aea04acec4601ecc,TR-1-018,base,closed,gpt-4o-mini,gpt-4o-mini,temporal_reasoning,easy,4,Türkçe dersi 09.20'de başlayacak ve 40 dakika ...,Sabah saat 08.30'da ilk ders zili çalan 3-A sı...,entailment,0.942781,0.009928,0.047292,entailment,0.997652,0.000486,0.001861
4,989752d2aea04acec4601ecc,TR-1-018,base,closed,gpt-4o-mini,gpt-4o-mini,temporal_reasoning,easy,5,"Türkçe dersi 09.20'de başladığı için, 40 dakik...",Sabah saat 08.30'da ilk ders zili çalan 3-A sı...,entailment,0.483157,0.415229,0.101614,entailment,0.552420,0.055820,0.391760
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,29262f68743a27d8076e247f,LR-2-008-v3,variants,open,qwen2.5-7b-instruct,Qwen/Qwen2.5-7B-Instruct,logical_inference,medium,3,Selin'e kesinlikle Biyolog görevi verilmemiş o...,Antarktika'daki bir araştırma üssünde 5 kişili...,neutral,0.018539,0.981066,0.000395,entailment,0.995073,0.003296,0.001631
222,29262f68743a27d8076e247f,LR-2-008-v3,variants,open,qwen2.5-7b-instruct,Qwen/Qwen2.5-7B-Instruct,logical_inference,medium,4,"Zorlu fiziksel şartlar gereği, Dalgıç, Teknisy...",Antarktika'daki bir araştırma üssünde 5 kişili...,entailment,0.981289,0.016318,0.002393,entailment,0.985497,0.007738,0.006765
223,29262f68743a27d8076e247f,LR-2-008-v3,variants,open,qwen2.5-7b-instruct,Qwen/Qwen2.5-7B-Instruct,logical_inference,medium,5,"Bu nedenle, kadınlar (Selin ve Elif) Biyolog, ...",Antarktika'daki bir araştırma üssünde 5 kişili...,entailment,0.998280,0.001526,0.000194,contradiction,0.003175,0.003661,0.993163
224,29262f68743a27d8076e247f,LR-2-008-v3,variants,open,qwen2.5-7b-instruct,Qwen/Qwen2.5-7B-Instruct,logical_inference,medium,6,"Can ve Mert erkekler olduğu için, Teknisyen ve...",Antarktika'daki bir araştırma üssünde 5 kişili...,entailment,0.996768,0.001321,0.001911,contradiction,0.135010,0.041155,0.823834



Saved evaluator preview files:
- results/evaluated/smoke_eval_v5/evaluator_preview/semantic_preview.csv
- results/evaluated/smoke_eval_v5/evaluator_preview/reasoning_nli_preview.csv
- results/evaluated/smoke_eval_v5/evaluator_preview/preview_metadata.json
